# Movie Review Bot - Google Colab

Tự động tạo video review phim YouTube sử dụng AI.

**Hướng dẫn:**
1. Chạy từng cell theo thứ tự
2. Nhập API keys khi được yêu cầu
3. Video sẽ được lưu vào Google Drive
4. Tải video về và thêm phụ đề trong CapCut

## Cell 1: Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ
import os
DRIVE_PATH = '/content/drive/MyDrive/Auto_review_youtube'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/output/videos', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/output/audio', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/output/subtitles', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/output/thumbnails', exist_ok=True)
print(f'Google Drive mounted. Files will be saved to: {DRIVE_PATH}')

## Cell 2: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update && apt-get install -y ffmpeg

# Install Python dependencies
!pip install edge-tts moviepy Pillow requests python-dotenv tqdm

print('Dependencies installed successfully!')

## Cell 3: Configure API Keys

Lấy API keys miễn phí:
- **DeepSeek API**: https://platform.deepseek.com (100K tokens/day)
- **OMDB API**: http://www.omdbapi.com/apikey.aspx (1,000 req/day, commercial use OK)

In [ ]:
import os

# Nhập API keys
# DeepSeek API (miễn phí, 100K tokens/day)
DEEPSEEK_API_KEY = input('Nhập DeepSeek API Key: ')

# OMDB API (miễn phí, 1,000 requests/day, cho phép thương mại)
OMDB_API_KEY = input('Nhập OMDB API Key: ')

# Cài đặt kênh
CHANNEL_LANGUAGE = input('Ngôn ngữ (vi hoặc en, mặc định: vi): ') or 'vi'
CHANNEL_NAME = input('Tên kênh YouTube (mặc định: PhimHay Review): ') or 'PhimHay Review'

# Lưu vào environment
os.environ['DEEPSEEK_API_KEY'] = DEEPSEEK_API_KEY
os.environ['OMDB_API_KEY'] = OMDB_API_KEY
os.environ['CHANNEL_LANGUAGE'] = CHANNEL_LANGUAGE
os.environ['CHANNEL_NAME'] = CHANNEL_NAME

print(f'\nCấu hình hoàn tất!')
print(f'Ngôn ngữ: {CHANNEL_LANGUAGE}')
print(f'Kênh: {CHANNEL_NAME}')

## Cell 4: Movie Data Fetcher

In [ ]:
import requests
from dataclasses import dataclass
from typing import Optional, List

OMDB_API_URL = 'http://www.omdbapi.com/'

@dataclass
class MovieData:
    title: str
    year: int
    overview: str
    rating: float
    imdb_rating: float
    rotten_tomatoes: str
    genre_names: List[str]
    cast_names: List[str]
    director: str
    poster_url: Optional[str]
    runtime: int
    language: str
    is_series: bool = False
    seasons: int = 0
    imdb_id: str = ''
    tmdb_id: int = 0
    backdrop_url: str = ''

def fetch_movie_by_title(title: str, year: int = 0) -> Optional[MovieData]:
    """Fetch movie details from OMDB using title."""
    api_key = os.environ.get('OMDB_API_KEY')
    if not api_key:
        print('Error: OMDB_API_KEY not set')
        return None

    params = {
        'apikey': api_key,
        't': title,
        'plot': 'full',
    }
    if year:
        params['y'] = year

    try:
        response = requests.get(OMDB_API_URL, params=params, timeout=30)
        data = response.json()

        if data.get('Response') == 'False':
            print(f'Error: {data.get("Error", "Movie not found")}')
            return None

        return _parse_omdb_data(data)

    except Exception as e:
        print(f'Error fetching movie: {e}')
        return None

def fetch_movie_by_imdb(imdb_id: str) -> Optional[MovieData]:
    """Fetch movie details from OMDB using IMDB ID."""
    api_key = os.environ.get('OMDB_API_KEY')
    if not api_key:
        print('Error: OMDB_API_KEY not set')
        return None

    params = {
        'apikey': api_key,
        'i': imdb_id,
        'plot': 'full',
    }

    try:
        response = requests.get(OMDB_API_URL, params=params, timeout=30)
        data = response.json()

        if data.get('Response') == 'False':
            print(f'Error: {data.get("Error", "Movie not found")}')
            return None

        return _parse_omdb_data(data)

    except Exception as e:
        print(f'Error fetching movie: {e}')
        return None

def search_movies(query: str, page: int = 1) -> list:
    """Search for movies on OMDB."""
    api_key = os.environ.get('OMDB_API_KEY')
    if not api_key:
        print('Error: OMDB_API_KEY not set')
        return []

    params = {
        'apikey': api_key,
        's': query,
        'type': 'movie',
        'page': page,
    }

    try:
        response = requests.get(OMDB_API_URL, params=params, timeout=30)
        data = response.json()

        if data.get('Response') == 'True':
            return data.get('Search', [])
        return []

    except Exception as e:
        print(f'Error searching: {e}')
        return []

def _parse_omdb_data(data: dict) -> MovieData:
    """Parse OMDB API response into MovieData."""
    # Parse genres
    genre_str = data.get('Genre', '')
    genre_names = [g.strip() for g in genre_str.split(',') if g.strip()]

    # Parse cast
    cast_str = data.get('Actors', '')
    cast_names = [c.strip() for c in cast_str.split(',') if c.strip()]

    # Parse year
    year_str = data.get('Year', '0')
    year = int(year_str[:4]) if year_str and year_str[:4].isdigit() else 0

    # Parse runtime
    runtime_str = data.get('Runtime', '0 min')
    runtime = int(runtime_str.replace(' min', '')) if 'min' in runtime_str else 0

    # Parse ratings
    imdb_rating = 0.0
    imdb_str = data.get('imdbRating', 'N/A')
    if imdb_str != 'N/A':
        try:
            imdb_rating = float(imdb_str)
        except:
            pass

    rotten_tomatoes = ''
    ratings = data.get('Ratings', [])
    for rating in ratings:
        if rating.get('Source') == 'Rotten Tomatoes':
            rotten_tomatoes = rating.get('Value', '')
            break

    # Check if series
    is_series = data.get('Type') == 'series'
    seasons = 0
    if is_series:
        total_seasons = data.get('totalSeasons', '0')
        seasons = int(total_seasons) if total_seasons.isdigit() else 0

    # Poster URL
    poster_url = data.get('Poster', '')
    if poster_url == 'N/A':
        poster_url = None

    movie = MovieData(
        title=data.get('Title', ''),
        year=year,
        overview=data.get('Plot', ''),
        rating=imdb_rating,
        imdb_rating=imdb_rating,
        rotten_tomatoes=rotten_tomatoes,
        genre_names=genre_names,
        cast_names=cast_names,
        director=data.get('Director', ''),
        poster_url=poster_url,
        runtime=runtime,
        language=data.get('Language', ''),
        is_series=is_series,
        seasons=seasons,
        imdb_id=data.get('imdbID', ''),
    )

    return movie

def download_image(url: str, save_path: str) -> bool:
    """Download image from URL."""
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return True
    except Exception as e:
        print(f'Error downloading image: {e}')
    return False

# Danh sách phim phổ biến (không cần API)
POPULAR_MOVIES = [
    'Fight Club', 'Pulp Fiction', 'Inception', 'The Dark Knight', 'Interstellar',
    'The Matrix', 'Forrest Gump', 'The Godfather', 'Schindler\'s List', 'The Shawshank Redemption',
    'Parasite', 'Joker', 'Avengers: Endgame', 'Spider-Man: No Way Home', 'Top Gun: Maverick',
]

print('Movie data module loaded! (Using OMDB API - commercial use allowed)')

## Cell 5: Script Generator (DeepSeek API)

In [ ]:
DEEPSEEK_API_URL = 'https://api.deepseek.com/v1/chat/completions'

# Script prompts
SCRIPT_PROMPTS = {
    'vi': {
        'movie': '''Bạn là một reviewer phim chuyên nghiệp trên YouTube. Hãy viết bài review phim sau bằng tiếng Việt.

THÔNG TIN PHIM:
- Tên phim: {title} ({year})
- Thể loại: {genres}
- Đạo diễn: {director}
- Diễn viên chính: {cast}
- Đánh giá TMDB: {rating}/10
- Nội dung: {overview}

YÊU CẦU:
1. Bài review phải dài ít nhất 2500 từ (tương đương 15+ phút nói)
2. Giọng văn tự nhiên, như đang nói chuyện với người xem
3. Cấu trúc:
   - Phần mở đầu: Hook hấp dẫn, giới thiệu phim
   - Phần nội dung: Phân tích cốt truyện, diễn xuất, kỹ thuật quay
   - Phần đánh giá: Đưa ra nhận xét công bằng
   - Phần kết: Tổng kết, đề xuất xem hay không
4. KHÔNG spoil quá nhiều nội dung phim
5. Thêm cảm xúc, câu hỏi tu từ để thu hút người xem
6. Đề cập đến điểm mạnh và điểm yếu của phim

Hãy viết bài review đầy đủ, chi tiết, và hấp dẫn.''',
        'series': '''Bạn là một reviewer phim bộ chuyên nghiệp trên YouTube. Hãy viết bài review phim bộ sau bằng tiếng Việt.

THÔNG TIN PHIM BỘ:
- Tên phim: {title} ({year})
- Season: {season}
- Số tập: {episodes}
- Thể loại: {genres}
- Đạo diễn: {director}
- Đánh giá TMDB: {rating}/10
- Nội dung: {overview}

YÊU CẦU:
1. Bài review phải dài ít nhất 2500 từ (tương đương 15+ phút nói)
2. Giọng văn tự nhiên, như đang nói chuyện với người xem
3. Cấu trúc:
   - Phần mở đầu: Hook hấp dẫn, giới thiệu phim bộ
   - Phần tóm tắt: Tóm tắt nội dung từng tập (không spoil quá nhiều)
   - Phần phân tích: Nhân vật, mối quan hệ, plot twist
   - Phần đánh giá: Đánh giá tổng thể season
   - Phần dự đoán: Dự đoán cho season tiếp theo
4. Phân tích sự phát triển của nhân vật qua các tập
5. Đề cập đến những khoảnh khắc ấn tượng nhất
6. So sánh với các phim bộ khác nếu có thể

Hãy viết bài review đầy đủ, chi tiết, và hấp dẫn.''',
    },
    'en': {
        'movie': '''You are a professional movie reviewer on YouTube. Write a detailed movie review in English.

MOVIE INFO:
- Title: {title} ({year})
- Genre: {genres}
- Director: {director}
- Cast: {cast}
- TMDB Rating: {rating}/10
- Overview: {overview}

REQUIREMENTS:
1. The review must be at least 2000 words (equivalent to 15+ minutes of speaking)
2. Natural, conversational tone - like talking to viewers
3. Structure:
   - Opening: Hook, introduce the movie
   - Content: Plot analysis, acting, cinematography
   - Evaluation: Fair assessment
   - Conclusion: Summary, recommendation
4. Don't spoil too much of the plot
5. Add emotions, rhetorical questions to engage viewers
6. Discuss strengths and weaknesses

Write a complete, detailed, and engaging review.''',
        'series': '''You are a professional TV series reviewer on YouTube. Write a detailed series review in English.

SERIES INFO:
- Title: {title} ({year})
- Season: {season}
- Episodes: {episodes}
- Genre: {genres}
- Creator: {director}
- TMDB Rating: {rating}/10
- Overview: {overview}

REQUIREMENTS:
1. The review must be at least 2000 words (equivalent to 15+ minutes of speaking)
2. Natural, conversational tone - like talking to viewers
3. Structure:
   - Opening: Hook, introduce the series
   - Summary: Episode breakdown (no major spoilers)
   - Analysis: Characters, relationships, plot twists
   - Evaluation: Overall season assessment
   - Predictions: Theories for next season
4. Analyze character development throughout episodes
5. Highlight the most impactful moments
6. Compare with similar series if possible

Write a complete, detailed, and engaging review.''',
    }
}

MIN_WORD_COUNT = {'vi': 2500, 'en': 2000}

def generate_review_script(movie: MovieData, language: str = None) -> Optional[str]:
    """Generate a movie review script using DeepSeek AI."""
    api_key = os.environ.get('DEEPSEEK_API_KEY')
    if not api_key:
        print('Error: DEEPSEEK_API_KEY not set')
        return None

    language = language or os.environ.get('CHANNEL_LANGUAGE', 'vi')
    content_type = 'series' if movie.is_series else 'movie'
    prompt_template = SCRIPT_PROMPTS[language][content_type]

    prompt = prompt_template.format(
        title=movie.title,
        year=movie.year,
        genres=', '.join(movie.genre_names),
        director=movie.director,
        cast=', '.join(movie.cast_names),
        rating=movie.rating,
        overview=movie.overview,
        season=movie.seasons if movie.is_series else '',
        episodes=movie.episodes if movie.is_series else '',
    )

    min_words = MIN_WORD_COUNT[language]
    prompt += f'\n\nIMPORTANT: The script must be at least {min_words} words to ensure 15+ minutes of speaking time.'

    try:
        headers = {
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {api_key}',
        }

        payload = {
            'model': 'deepseek-chat',
            'messages': [
                {
                    'role': 'system',
                    'content': 'You are a professional movie reviewer for YouTube. Write detailed, engaging reviews in a natural, conversational tone.'
                },
                {
                    'role': 'user',
                    'content': prompt,
                }
            ],
            'temperature': 0.8,
            'max_tokens': 8192,
            'stream': False,
        }

        response = requests.post(DEEPSEEK_API_URL, headers=headers, json=payload, timeout=120)

        if response.status_code == 200:
            data = response.json()
            script = data['choices'][0]['message']['content']
            word_count = len(script.split())
            print(f'Generated script: {word_count} words (target: {min_words}+ words)')
            return script
        else:
            print(f'Error: DeepSeek API returned {response.status_code}')
            print(response.text)
            return None

    except Exception as e:
        print(f'Error generating script: {e}')
        return None

print('Script generator module loaded!')

## Cell 6: Voice Generator (Edge TTS)

In [ ]:
import edge_tts
import asyncio

VOICES = {
    'vi': {
        'female': 'vi-VN-HoaiMyNeural',
        'male': 'vi-VN-NamMinhNeural',
    },
    'en': {
        'female': 'en-US-JennyNeural',
        'male': 'en-US-GuyNeural',
    },
}

async def generate_audio(script: str, output_path: str, language: str = 'vi', voice_gender: str = 'female') -> bool:
    """Generate audio from script using Edge TTS."""
    voice = VOICES.get(language, VOICES['vi']).get(voice_gender, VOICES['vi']['female'])

    try:
        communicate = edge_tts.Communicate(script, voice)
        await communicate.save(output_path)
        print(f'Audio generated: {output_path}')
        return True
    except Exception as e:
        print(f'Error generating audio: {e}')
        return False

def generate_movie_audio(script: str, tmdb_id: int, language: str = 'vi') -> Optional[str]:
    """Generate audio for a movie review."""
    output_path = f'{DRIVE_PATH}/output/audio/{tmdb_id}.mp3'
    success = asyncio.run(generate_audio(script, output_path, language))
    return output_path if success else None

print('Voice generator module loaded!')

## Cell 7: Subtitle Generator

In [ ]:
def generate_subtitles(script: str, tmdb_id: int, language: str = 'vi') -> Optional[str]:
    """Generate SRT subtitles from script."""
    output_path = f'{DRIVE_PATH}/output/subtitles/{tmdb_id}.srt'

    # Split script into sentences
    sentences = []
    for paragraph in script.split('\n\n'):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        # Split by sentence endings
        for sentence in paragraph.replace('. ', '.\n').replace('? ', '?\n').replace('! ', '!\n').split('\n'):
            sentence = sentence.strip()
            if sentence and len(sentence) > 10:
                sentences.append(sentence)

    if not sentences:
        print('Error: No sentences found in script')
        return None

    # Calculate timing
    wpm = 150 if language == 'vi' else 160
    total_words = len(script.split())
    total_duration = (total_words / wpm) * 60  # in seconds
    avg_duration_per_sentence = total_duration / len(sentences)

    # Generate SRT
    srt_content = ''
    current_time = 0

    for i, sentence in enumerate(sentences, 1):
        # Calculate duration based on word count
        words = len(sentence.split())
        duration = max(2, (words / wpm) * 60)  # At least 2 seconds

        # Format timestamps
        start_h = int(current_time // 3600)
        start_m = int((current_time % 3600) // 60)
        start_s = int(current_time % 60)
        start_ms = int((current_time % 1) * 1000)

        end_time = current_time + duration
        end_h = int(end_time // 3600)
        end_m = int((end_time % 3600) // 60)
        end_s = int(end_time % 60)
        end_ms = int((end_time % 1) * 1000)

        srt_content += f'{i}\n'
        srt_content += f'{start_h:02d}:{start_m:02d}:{start_s:02d},{start_ms:03d} --> '
        srt_content += f'{end_h:02d}:{end_m:02d}:{end_s:02d},{end_ms:03d}\n'
        srt_content += f'{sentence}\n\n'

        current_time = end_time

    # Save SRT file
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(srt_content)

    print(f'Subtitles generated: {len(sentences)} sentences, {current_time:.1f} seconds')
    return output_path

print('Subtitle generator module loaded!')

## Cell 8: Thumbnail Generator

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import io

def generate_thumbnail(movie: MovieData, language: str = 'vi') -> Optional[str]:
    """Generate thumbnail for YouTube video."""
    output_path = f'{DRIVE_PATH}/output/thumbnails/{movie.tmdb_id}.jpg'

    try:
        # Download poster
        if movie.poster_url:
            response = requests.get(movie.poster_url, timeout=30)
            if response.status_code == 200:
                poster = Image.open(io.BytesIO(response.content))
            else:
                poster = Image.new('RGB', (1280, 720), color=(30, 30, 30))
        else:
            poster = Image.new('RGB', (1280, 720), color=(30, 30, 30))

        # Resize to 1280x720 (YouTube thumbnail size)
        thumbnail = poster.resize((1280, 720), Image.Resampling.LANCZOS)

        # Add overlay
        overlay = Image.new('RGBA', (1280, 720), (0, 0, 0, 128))
        thumbnail = Image.alpha_composite(thumbnail.convert('RGBA'), overlay)

        # Add text
        draw = ImageDraw.Draw(thumbnail)

        # Title text
        title_text = f'REVIEW: {movie.title}'
        try:
            font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 48)
        except:
            font = ImageFont.load_default()

        # Draw title with outline
        bbox = draw.textbbox((0, 0), title_text, font=font)
        text_width = bbox[2] - bbox[0]
        x = (1280 - text_width) // 2
        y = 50

        # Outline
        for dx, dy in [(-2, -2), (-2, 2), (2, -2), (2, 2)]:
            draw.text((x+dx, y+dy), title_text, font=font, fill=(0, 0, 0))
        # Main text
        draw.text((x, y), title_text, font=font, fill=(255, 255, 255))

        # Rating text
        rating_text = f'{movie.rating}/10'
        try:
            rating_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 64)
        except:
            rating_font = ImageFont.load_default()

        bbox = draw.textbbox((0, 0), rating_text, font=rating_font)
        text_width = bbox[2] - bbox[0]
        x = (1280 - text_width) // 2
        y = 600

        # Outline
        for dx, dy in [(-2, -2), (-2, 2), (2, -2), (2, 2)]:
            draw.text((x+dx, y+dy), rating_text, font=rating_font, fill=(0, 0, 0))
        # Main text (gold color)
        draw.text((x, y), rating_text, font=rating_font, fill=(255, 215, 0))

        # Save
        thumbnail.convert('RGB').save(output_path, 'JPEG', quality=95)
        print(f'Thumbnail generated: {output_path}')
        return output_path

    except Exception as e:
        print(f'Error generating thumbnail: {e}')
        return None

print('Thumbnail generator module loaded!')

## Cell 9: Video Builder

In [ ]:
from moviepy.editor import (
    ImageClip, AudioFileClip, TextClip, CompositeVideoClip,
    concatenate_videoclips, ColorClip
)
import numpy as np

def build_video(
    movie: MovieData,
    audio_path: str,
    subtitle_path: str,
    thumbnail_path: Optional[str] = None,
    language: str = 'vi',
    background_images: list = None,
) -> Optional[str]:
    """Build final video from components."""
    output_path = f'{DRIVE_PATH}/output/videos/{movie.tmdb_id}.mp4'

    try:
        # Load audio
        audio = AudioFileClip(audio_path)
        duration = audio.duration

        print(f'Audio duration: {duration:.1f} seconds ({duration/60:.1f} minutes)')

        # Create background clips
        clips = []
        if background_images and len(background_images) > 0:
            # Use background images
            img_duration = duration / len(background_images)
            for img_path in background_images:
                try:
                    img = Image.open(img_path)
                    img = img.resize((1920, 1080), Image.Resampling.LANCZOS)
                    img_array = np.array(img)
                    clip = ImageClip(img_array).set_duration(img_duration)
                    clips.append(clip)
                except Exception as e:
                    print(f'Warning: Could not load image {img_path}: {e}')

        if not clips:
            # Fallback: solid color background
            bg = ColorClip(size=(1920, 1080), color=(30, 30, 30)).set_duration(duration)
            clips = [bg]

        # Concatenate background clips
        video = concatenate_videoclips(clips, method='compose')

        # Set audio
        video = video.set_audio(audio)

        # Add movie title overlay
        title_text = f'{movie.title} ({movie.year})'
        try:
            title_clip = TextClip(
                title_text,
                fontsize=60,
                color='white',
                font='Arial',
                stroke_color='black',
                stroke_width=2,
            ).set_position(('center', 50)).set_duration(min(10, duration))

            video = CompositeVideoClip([video, title_clip])
        except Exception as e:
            print(f'Warning: Could not add title overlay: {e}')

        # Export video
        print('Exporting video...')
        video.write_videofile(
            output_path,
            fps=30,
            codec='libx264',
            audio_codec='aac',
            threads=4,
            logger=None,
        )

        print(f'Video saved: {output_path}')
        return output_path

    except Exception as e:
        print(f'Error building video: {e}')
        return None

print('Video builder module loaded!')

## Cell 10: Main Pipeline

In [ ]:
import time

def process_movie(
    tmdb_id: int,
    media_type: str = 'movie',
    language: str = None,
) -> Optional[str]:
    """Process a single movie: fetch data, generate script, build video."""
    language = language or os.environ.get('CHANNEL_LANGUAGE', 'vi')
    start_time = time.time()

    print(f'\n{"="*60}')
    print(f'Processing Movie ID: {tmdb_id}')
    print(f'Language: {language}')
    print(f'{"="*60}\n')

    # Step 1: Fetch movie data
    print('[1/6] Fetching movie data...')
    movie = fetch_movie_details(tmdb_id, media_type)
    if not movie:
        print('Error: Failed to fetch movie data')
        return None

    print(f'  Title: {movie.title} ({movie.year})')
    print(f'  Rating: {movie.rating}/10')
    print(f'  Genres: {", ".join(movie.genre_names)}')

    # Step 2: Generate review script
    print('\n[2/6] Generating review script...')
    script = generate_review_script(movie, language)
    if not script:
        print('Error: Failed to generate script')
        return None

    word_count = len(script.split())
    print(f'  Script generated: {word_count} words')

    # Save script
    script_path = f'{DRIVE_PATH}/output/{tmdb_id}_script.txt'
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(script)
    print(f'  Script saved: {script_path}')

    # Step 3: Generate audio
    print('\n[3/6] Generating audio...')
    audio_path = generate_movie_audio(script, tmdb_id, language)
    if not audio_path:
        print('Error: Failed to generate audio')
        return None

    # Step 4: Generate subtitles
    print('\n[4/6] Generating subtitles...')
    subtitle_path = generate_subtitles(script, tmdb_id, language)
    if not subtitle_path:
        print('Error: Failed to generate subtitles')
        return None

    # Step 5: Generate thumbnail
    print('\n[5/6] Generating thumbnail...')
    thumbnail_path = generate_thumbnail(movie, language)

    # Step 6: Download background images
    print('\n[6/6] Downloading background images...')
    background_images = []
    if movie.poster_url:
        poster_path = f'{DRIVE_PATH}/output/thumbnails/{movie.tmdb_id}_poster.jpg'
        if download_image(movie.poster_url, poster_path):
            background_images.append(poster_path)
    if movie.backdrop_url:
        backdrop_path = f'{DRIVE_PATH}/output/thumbnails/{movie.tmdb_id}_backdrop.jpg'
        if download_image(movie.backdrop_url, backdrop_path):
            background_images.append(backdrop_path)
    print(f'  Downloaded {len(background_images)} background images')

    # Build video
    print('\n[BUILD] Building video...')
    video_path = build_video(
        movie=movie,
        audio_path=audio_path,
        subtitle_path=subtitle_path,
        thumbnail_path=thumbnail_path,
        language=language,
        background_images=background_images,
    )

    if not video_path:
        print('Error: Failed to build video')
        return None

    # Summary
    elapsed = time.time() - start_time
    print(f'\n{"="*60}')
    print(f'Processing complete!')
    print(f'Time elapsed: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
    print(f'Video saved to: {video_path}')
    print(f'Subtitles saved to: {subtitle_path}')
    print(f'{"="*60}\n')

    return video_path

print('Pipeline module loaded!')

## Cell 11: Run the Bot!

Chọn 1 trong 2 cách bên dưới để chạy:

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÁCH 1: Chạy với 1 phim cụ thể (theo tên)
# ═══════════════════════════════════════════════════════════

# Nhập tên phim muốn review
MOVIE_TITLE = 'Fight Club'  # Thay đổi tên phim tại đây

# Tìm kiếm phim
print(f'Tìm kiếm phim: {MOVIE_TITLE}')
movie = fetch_movie_by_title(MOVIE_TITLE)

if movie:
    print(f'\nTìm thấy: {movie.title} ({movie.year})')
    print(f'Rating: {movie.rating}/10')
    print(f'IMDB: {movie.imdb_rating}/10')
    print(f'Rotten Tomatoes: {movie.rotten_tomatoes}')
    print(f'Director: {movie.director}')
    print(f'Cast: {", ".join(movie.cast_names)}')
    print(f'Genres: {", ".join(movie.genre_names)}')

    # Tạo ID từ tên phim (để lưu file)
    movie_id = hash(movie.title) % 100000

    # Chạy pipeline
    script = generate_review_script(movie, os.environ.get('CHANNEL_LANGUAGE', 'vi'))

    if script:
        # Lưu script
        script_path = f'{DRIVE_PATH}/output/{movie_id}_script.txt'
        with open(script_path, 'w', encoding='utf-8') as f:
            f.write(script)
        print(f'\nScript saved: {script_path}')

        # Generate audio
        audio_path = generate_movie_audio(script, movie_id, os.environ.get('CHANNEL_LANGUAGE', 'vi'))

        if audio_path:
            # Generate subtitles
            subtitle_path = generate_subtitles(script, movie_id, os.environ.get('CHANNEL_LANGUAGE', 'vi'))

            # Generate thumbnail
            thumbnail_path = generate_thumbnail(movie, os.environ.get('CHANNEL_LANGUAGE', 'vi'))

            # Download poster for background
            background_images = []
            if movie.poster_url:
                poster_path = f'{DRIVE_PATH}/output/thumbnails/{movie_id}_poster.jpg'
                if download_image(movie.poster_url, poster_path):
                    background_images.append(poster_path)

            # Build video
            video_path = build_video(
                movie=movie,
                audio_path=audio_path,
                subtitle_path=subtitle_path,
                thumbnail_path=thumbnail_path,
                language=os.environ.get('CHANNEL_LANGUAGE', 'vi'),
                background_images=background_images,
            )

            if video_path:
                print(f'\n✅ Video created successfully!')
                print(f'📁 Check Google Drive: {video_path}')
            else:
                print(f'\n❌ Failed to create video')
        else:
            print(f'\n❌ Failed to generate audio')
    else:
        print(f'\n❌ Failed to generate script')
else:
    print(f'\n❌ Movie not found: {MOVIE_TITLE}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÁCH 2: Chạy với danh sách phim phổ biến
# ═══════════════════════════════════════════════════════════

# Danh sách phim phổ biến để review
MOVIE_LIST = [
    'Fight Club',
    'Pulp Fiction',
    'Inception',
    'The Dark Knight',
    'Interstellar',
    # Thêm phim khác tại đây
]

results = []
for i, movie_title in enumerate(MOVIE_LIST):
    print(f'\nProcessing {i+1}/{len(MOVIE_LIST)}: {movie_title}')

    movie = fetch_movie_by_title(movie_title)
    if not movie:
        print(f'Skipping: {movie_title} not found')
        continue

    print(f'  Found: {movie.title} ({movie.year}) - Rating: {movie.rating}/10')

    # Tạo ID từ tên phim
    movie_id = hash(movie.title) % 100000

    # Chạy pipeline
    script = generate_review_script(movie, os.environ.get('CHANNEL_LANGUAGE', 'vi'))
    if not script:
        print(f'  Skipping: Failed to generate script')
        continue

    # Lưu script
    script_path = f'{DRIVE_PATH}/output/{movie_id}_script.txt'
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(script)

    # Generate audio
    audio_path = generate_movie_audio(script, movie_id, os.environ.get('CHANNEL_LANGUAGE', 'vi'))
    if not audio_path:
        print(f'  Skipping: Failed to generate audio')
        continue

    # Generate subtitles
    subtitle_path = generate_subtitles(script, movie_id, os.environ.get('CHANNEL_LANGUAGE', 'vi'))

    # Generate thumbnail
    thumbnail_path = generate_thumbnail(movie, os.environ.get('CHANNEL_LANGUAGE', 'vi'))

    # Download poster
    background_images = []
    if movie.poster_url:
        poster_path = f'{DRIVE_PATH}/output/thumbnails/{movie_id}_poster.jpg'
        if download_image(movie.poster_url, poster_path):
            background_images.append(poster_path)

    # Build video
    video_path = build_video(
        movie=movie,
        audio_path=audio_path,
        subtitle_path=subtitle_path,
        thumbnail_path=thumbnail_path,
        language=os.environ.get('CHANNEL_LANGUAGE', 'vi'),
        background_images=background_images,
    )

    if video_path:
        results.append(video_path)
        print(f'  ✅ Video created: {video_path}')
    else:
        print(f'  ❌ Failed to create video')

print(f'\n{"="*60}')
print(f'Batch processing complete!')
print(f'Created {len(results)}/{len(MOVIE_LIST)} videos')
for path in results:
    print(f'  - {path}')
print(f'{"="*60}')

## Cell 12: Batch Processing (Optional)

Tạo nhiều video cùng lúc:

In [ ]:
# Batch processing - Tạo nhiều video

# Danh sách phim muốn review
MOVIE_LIST = [
    {'id': 550, 'type': 'movie', 'name': 'Fight Club'},
    {'id': 680, 'type': 'movie', 'name': 'Pulp Fiction'},
    {'id': 27205, 'type': 'movie', 'name': 'Inception'},
    # Thêm phim khác tại đây
]

results = []
for i, movie_info in enumerate(MOVIE_LIST):
    print(f'\nProcessing {i+1}/{len(MOVIE_LIST)}: {movie_info["name"]}')
    video_path = process_movie(
        tmdb_id=movie_info['id'],
        media_type=movie_info['type'],
        language=os.environ.get('CHANNEL_LANGUAGE', 'vi'),
    )
    if video_path:
        results.append(video_path)

print(f'\n{"="*60}')
print(f'Batch processing complete!')
print(f'Created {len(results)}/{len(MOVIE_LIST)} videos')
for path in results:
    print(f'  - {path}')
print(f'{"="*60}')

## Cell 13: List Created Videos

In [ ]:
# List all created videos
import glob

videos = glob.glob(f'{DRIVE_PATH}/output/videos/*.mp4')
subtitles = glob.glob(f'{DRIVE_PATH}/output/subtitles/*.srt')
thumbnails = glob.glob(f'{DRIVE_PATH}/output/thumbnails/*.jpg')

print(f'📁 Videos: {len(videos)}')
for v in videos:
    size_mb = os.path.getsize(v) / (1024*1024)
    print(f'  - {os.path.basename(v)} ({size_mb:.1f} MB)')

print(f'\n📝 Subtitles: {len(subtitles)}')
for s in subtitles:
    print(f'  - {os.path.basename(s)}')

print(f'\n🖼️ Thumbnails: {len(thumbnails)}')
for t in thumbnails:
    print(f'  - {os.path.basename(t)}')

## Cell 14: Download Video to Local

Tải video về máy để chỉnh sửa trong CapCut

In [ ]:
# Download video from Drive to local
from google.colab import files

# Chọn video muốn tải
video_files = glob.glob(f'{DRIVE_PATH}/output/videos/*.mp4')

if video_files:
    print('Videos available:')
    for i, v in enumerate(video_files):
        print(f'  {i+1}. {os.path.basename(v)}')

    # Tải video đầu tiên
    selected_video = video_files[0]
    print(f'\nDownloading: {os.path.basename(selected_video)}')
    files.download(selected_video)

    # Tải subtitles
    subtitle_files = glob.glob(f'{DRIVE_PATH}/output/subtitles/*.srt')
    if subtitle_files:
        print(f'\nDownloading subtitles...')
        files.download(subtitle_files[0])
else:
    print('No videos found. Run the pipeline first!')

## Hướng dẫn hậu kỳ với CapCut

### Bước 1: Import video
1. Mở CapCut
2. Tạo project mới (1920x1080, 30fps)
3. Import video vừa tải

### Bước 2: Thêm phụ đề
1. Click **Text** > **Auto Captions**
2. Chọn ngôn ngữ phù hợp
3. CapCut sẽ tự动生成 phụ đề từ audio
4. Hoặc import file .srt đã tải

### Bước 3: Chỉnh sửa
1. Thêm intro/outro
2. Chỉnh màu sắc, hiệu ứng
3. Thêm nhạc nền (royalty-free)
4. Thêm subscribe button

### Bước 4: Export
1. Export ở 1080p, 30fps
2. Upload lên YouTube
3. Thêm title, description, tags

### Mẹo:
- Dùng **Auto Caption** thay vì import SRT (chính xác hơn)
- Thêm **zoom effects** để video sinh động
- Dùng **background music** volume 10-20%
- Thêm **end screen** với video suggestion